In [2]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

/opt/venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# Udahub Application

## Core Database

**Init DB**

In [3]:
udahub_db = "data/core/udahub.db"

In [8]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-09-05 14:29:31,030 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-05 14:29:31,030 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [9]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [10]:
account_id = "cultpass"
account_name = "CultPass Card"

In [11]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [12]:
# TODO: Create additional 10 articles
# Added them in the file directly

In [13]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [14]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [15]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [16]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [17]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [18]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [19]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [20]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [21]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='2ddbfc14-aff3-47d4-96ba-1904e7f3178e', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='d76425aa-78a3-4ca2-9f6b-5c9eb8b65cff', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='9bfc7b94-2884-46f8-acd8-5ea4fc1aeaf2', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='ba619d97-9a49-4ea1-923c-2829db9b458e', title='How to Handle Login Issues?')>
<Knowledge(article_id='ec138182-8304-4d2a-83db-97ab6386fad7', title='How to Check Remaining Monthly Experiences')>
<Knowledge(article_id='293bd54c-72e7-4f37-adc5-083b8ba0fee5', title='What to Do If a QR Code Is Not Showing')>
<Knowledge(article_id='f7da7e8d-99a8-423e-8152-e3586a7f2cfb', title='How to Cancel an Individual Experience Reservation')>
<Knowledge(article_id='2aec2ff4-e6eb-4bdf-ba80-d5e15a204ac5', title='How to Reschedule an Experience')>
<Knowledge(article_id='39238bbf-9dcc-4995-a9a2-0401d9682cb6', title='What to Do If an Experience Is Fully Booked')>

In [22]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='5fa4f412-e93f-47f5-8e96-4a2cd1aab33c', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [23]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='f7b945cb-04a7-42b0-b799-9012b9225708', role='user', content='I can't log in to my Cultpass ...')>


In [24]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message.role.value, message.role)
        role = message.role
        print(type(role), role, role.value if hasattr(role, 'value') else None)

user RoleEnum.user
<enum 'RoleEnum'> RoleEnum.user user


In [20]:
with get_session(engine) as session:
    ticket_messages = session.query(udahub.TicketMessage).all()
    for msg in ticket_messages:
        print(msg.content)

I can't log in to my Cultpass account.
